# DuckDB
https://duckdb.org

### 우리가 지금까지 배운 것들
- `Numpy` 데이터 분석과 인공지능을 위한 데이터 계산 엔진
- `Pandas` 흩어진 데이터를 자르고, 붙이고, 그룹별로 통계량을 뽑는 분석적 사고의 뼈대

- `Matplotlib & Seaborn` 숫자로 가득 찬 데이터에서 유의미한 패턴을 찾아내는 시각화 감각
- `scikit-learn` 과거의 패턴을 바탕으로 미래를 예측하는 머신러닝 파이프라인

### 이상과 현실
- 이상 : 기계에서 엑셀 파일 생성 -> CSV 변환 -> 데이터 분석 -> 보고서 작성
- 현실 : 기계에서 DB 저장 -> CSV 생성 -> 데이터 분석 -> 보고서 작성 (그리고 CSV를 바로 DB에 적용) 

- DB에서 데이터를 다루는 가장 일반적인 방법 : SQL
- 파이썬 데이터 분석에서 SQL을 바로 적용하는 방법 : DuckDB 

### 실무에서 발생하는 일들

현업이나 대규모 프로젝트에서 수백만, 수천만 건의 데이터를 다루다 보면 pandas만으로는 마주치는 명확한 한계

- 💥 **메모리 폭발 (OOM, Out Of Memory):** "조금 큰 CSV를 읽었을 뿐인데 주피터 커널이 죽어버려요."
  * pandas는 파일 크기의 약 **5~10배에 달하는 RAM**을 요구

- ⏳ **답답한 단일 코어 연산:** 내 노트북에는 8코어, 16코어 최신 CPU가 있는데, pandas는 대부분 **1개 코어**만 쓰며 느리게 돌아갑니다.

- 💾 **거대한 로컬 DB의 진입 장벽:** 대용량 데이터를 처리하려고 PostgreSQL, MySQL을 설치하자니 서버 세팅과 포트 관리에 진이 빠집니다.


### DuckDB란?

**DuckDB**는 복잡한 서버 설정 없이, Python 코드 안에서 즉시 실행되는 **초경량 초고속 OLAP(분석용) SQL 엔진**입니다.

- ⚡ **압도적인 속도:** 최신 CPU의 멀티코어를 100% 활용하는 **벡터화(Vectorized) 엔진** 탑재

- 🔄 **Zero-Copy 연동:** 여러분이 쓰던 **pandas DataFrame을 변환 없이 그대로 SQL 테이블처럼 조회**

- 🛡️ **메모리 절약 (Streaming):** RAM보다 큰 대용량 파일(Parquet/CSV)도 노트북이 튕기지 않고 필요한 부분만 스캔

- 📦 **극강의 간편함:** 복잡한 DB 서버 설치 없이 `pip install duckdb` 단 한 줄로 완료

### 우리가 배운 pandas vs DuckDB SQL 1:1 매칭

새로운 언어를 처음부터 배울 필요가 전혀 없습니다. pandas의 로직을 SQL로 바꿔 부르기만 하면 됩니다.

| 분석 작업 | pandas 문법 | DuckDB SQL 문법 |
| :--- | :--- | :--- |
| **컬럼 선택 및 계산** | `df[['user_id', 'amount']]` | `SELECT user_id, amount FROM df` |
| **조건 필터링** | `df[df['amount'] >= 10000]` | `WHERE amount >= 10000` |
| **그룹별 요약** | `df.groupby('grade')['amount'].mean()` | `SELECT grade, AVG(amount) GROUP BY grade` |
| **정렬** | `df.sort_values(by='amount', ascending=False)` | `ORDER BY amount DESC` |
| **테이블 결합** | `pd.merge(df1, df2, on='id', how='left')` | `FROM df1 LEFT JOIN df2 ON df1.id = df2.id` |

```
"무겁고 거대한 데이터 필터링/집계는 DuckDB로 0.1초 만에 끝내고, 결과는 우리가 익숙한 pandas DataFrame으로 돌려받아 Random Forest로 모델링한다!"
```

# 시작하기
DuckDB 엔진 설치
```
pip install duckdb
```

# SQL부터 천천히 배워봅시다

우선 간단한 Pandas DataFrame 데이터를 만들어봅시다.

In [1]:
# 코드

전체 조회와 컬럼 선택 (SELECT, FROM)
- pandas에서 컬럼을 뽑아보던 df[['name', 'score']]와 대응됩니다.

In [2]:
# 코드

새로운 컬럼 계산과 별칭 주기 (AS)
- pandas에서 df['new_col'] = df['a'] + df['b'] 하던 작업을 AS 새컬럼명으로 처리합니다.

In [3]:
# 코드

조건 필터링 (WHERE)
- pandas의 불리언 인덱싱 df[df['score'] >= 85] 또는 df.query(...)와 대응됩니다.

In [4]:
# 코드

정렬하기 (ORDER BY)
- pandas의 df.sort_values(by='score', ascending=False)와 대응됩니다.

In [5]:
# 코드

통계 요약과 그룹화 (GROUP BY, 집계 함수)
- pandas의 df.groupby('class')['score'].mean()과 대응되는 가장 중요한 핵심 단계입니다.

In [6]:
# 코드

조건문 분기 (CASE WHEN)
- pandas의 np.where()나 apply(lambda ...) 역할을 하는 SQL의 조건문입니다.

In [7]:
# 코드

# SQL 문장의 순서 읽는 법
1. FROM(어디서) → 
1. WHERE(누구를) → 
1. GROUP BY(묶어서) → 
1. SELECT(무엇을 뽑아서) → 
1. ORDER BY(정렬할까)

### CMAPSS 읽기
한 줄로 읽어서 pandas DataFrame으로 가져오기

In [8]:
# 코드

읽으면서 바로 필터링 & 집계하기 (DuckDB의 핵심 강점)
- `pd.read_csv()`처럼 전체 파일을 RAM에 다 올린 뒤 필터링하지 않고, 디스크에서 조건에 맞는 데이터만 골라 메모리로 가져오는 방식입니다.

In [9]:
# 코드

세션(Connection)을 열고 정식 테이블/뷰로 등록해서 다루기
- 여러 번 쿼리를 할 때는 연결 객체(con)를 만들어 VIEW로 등록해두면 파일 경로를 매번 적지 않아도 되어 편리합니다.

In [10]:
# 코드

시각화해봅시다

In [11]:
# 코드

# 이번엔 MIMII를 읽어봅시다

In [12]:
# 코드

음향 피처 요약 집계 (정상 vs 이상 음향 통계 비교)
- 음향 분석의 핵심 지표(진폭 실효값 RMS, 스펙트럼 중심, 영교차율)가 정상(label=0)과 이상(label=1) 상태에서 어떻게 다른지 SQL의 집계 함수(AVG, STDDEV, MIN, MAX)로 한 번에 요약합니다.

In [13]:
# 코드

이상치 후보 추출 및 조건 필터링 (Feature Screening)
- 이상 음향의 특성(진폭이 크거나 주파수 중심이 높은 패턴)에 해당하는 샘플만 디스크 레벨에서 빠르게 필터링해 가져옵니다.

In [14]:
# 코드

시각화(seaborn)로 바로 연결하기
- 추출한 데이터를 기존에 배운 seaborn에 넣어 정상과 이상의 음향 특징 차이를 시각화합니다.

In [15]:
# 코드

# DuckDB와 Pandas의 DataFrame은 완전히 호환됩니다
- pandas 고유 메서드: .info(), .describe(), .iloc, .loc, .apply() 등 즉시 사용 가능
- 시각화: seaborn, matplotlib의 data= 파라미터로 직행
- 머신러닝: scikit-learn의 fit(X, y)나 train_test_split()에 전달 가능

In [16]:
# 코드

기능은 100% 호환되지만, 데이터 타입 매핑 과정에서 생길 수 있는 차이점
- 정수형 결측치 처리:
    - pandas 순정 read_csv는 결측치(NULL)가 섞인 정수 컬럼을 자동으로 float64로 바꾸는 경향이 있습니다.
    - DuckDB의 .df()는 내부적으로 pandas의 최신 Nullable 타입(Int64) 등을 적극 활용하거나 화살표 확장을 써서 정수 형태를 온전히 보존해 반환합니다.

- 시간대(Timestamp) 처리:
    - 날짜/시간 컬럼의 세부 단위(초, 밀리초, 마이크로초)나 타임존 속성이 DuckDB의 원본 규격에 맞춰 datetime64[us] 등으로 정밀하게 매핑됩니다.
